# 選定テーマのドメイン・サービス設計（2_03）

Notebook 2_02で選んだ`admin_process × ai_usecase`について、**どのpolicy domainを優先し、どの省庁へ、どのようなサービス・コンサルティングを提供するか**を検討します。

件数・率・予算・省庁・年度・課題文は観測データです。一方、提供サービス、PoC、実装支援、KPI、リスクは編集可能な仮説カタログです。両者を分離して表示し、データからサービス案が自動的に証明されたように扱いません。LLM APIは呼びません。

In [ ]:
from pathlib import Path
import sys

import japanize_matplotlib  # noqa: F401
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from llm_features import ADMIN_PROCESSES, AI_USECASES, POLICY_DOMAINS
from market_analysis import build_theme_long, merge_project_taxonomy
from service_analysis import (
    ADMIN_CONSULTING_CATALOG,
    DOMAIN_OPPORTUNITY_WEIGHTS,
    USECASE_SERVICE_CATALOG,
    build_service_opportunity_cards,
    compute_ministry_domain_opportunities,
    compute_theme_domain_metrics,
    compute_theme_domain_yearly_metrics,
    extract_distinctive_char_ngrams,
    score_theme_domain_metrics,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)

def save_figure(name: str) -> None:
    path = OUTPUT_DIR / name
    plt.savefig(path, dpi=180, bbox_inches='tight')
    print('saved:', path)

## 1. 設定

`SELECTED_THEME_CODES`に2_02で決めたテーマを指定します。空listなら2_02の総合順位上位から自動選択します。予算列が存在しない、または全欠損の場合は、その15%を利用可能な指標へ比例再配分します。

In [ ]:
TRAIN_PATH = PROJECT_ROOT / 'input' / 'train.csv'
TEST_PATH = PROJECT_ROOT / 'input' / 'test.csv'
LLM_FEATURE_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_llm_features.csv.gz'
THEME_METRICS_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_theme_metrics.csv'
DOMAIN_METRICS_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_theme_domain_metrics.csv'
DOMAIN_YEARLY_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_theme_domain_yearly_metrics.csv'
MINISTRY_OPPORTUNITY_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_ministry_domain_opportunities.csv'
PHRASE_PATH = PROJECT_ROOT / 'data' / 'csv' / 'ai_market_distinctive_phrases.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'ai_market_service_design'

ID_COL = 'project_id'
YEAR_COL = 'project_start_year'
MINISTRY_COL = 'responsible_ministry'
BUDGET_COL = 'budget'
ISSUE_COL = 'current_issues'
MIN_YEAR = 2020

SELECTED_THEME_CODES = []  # 例: ['A03__U05', 'A06__U06']
AUTO_TOP_THEME_N = 3
TOP_DOMAINS_PER_THEME = 3
RECENT_YEARS = 2
RATE_PRIOR_STRENGTH = 10.0
RUN_TEXT_PHRASE_ANALYSIS = True
TOP_PHRASES = 15
MAX_PHRASE_PAIRS = 6

DOMAIN_WEIGHTS = DOMAIN_OPPORTUNITY_WEIGHTS.copy()
ADMIN_SERVICE_RULES = ADMIN_CONSULTING_CATALOG.copy()
USECASE_SERVICE_RULES = USECASE_SERVICE_CATALOG.copy()

display(pd.Series(DOMAIN_WEIGHTS, name='configured_weight').to_frame())

## 2. データ読込・選定テーマ確認

2_01のLLM分類、2_02のテーマ順位、元のtrain/testが必要です。2_03はこれらを再利用し、APIを呼びません。

In [ ]:
required_files = [TRAIN_PATH, TEST_PATH, LLM_FEATURE_PATH, THEME_METRICS_PATH]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Notebook 2_01と2_02を先に実行してください。missing: '
        + ', '.join(str(path) for path in missing_files)
    )

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
llm_features = pd.read_csv(LLM_FEATURE_PATH, compression='gzip')
theme_ranking = pd.read_csv(THEME_METRICS_PATH)
projects, merge_diagnostics = merge_project_taxonomy(
    train, test, llm_features,
    id_col=ID_COL, year_col=YEAR_COL, ministry_col=MINISTRY_COL, min_year=MIN_YEAR,
)
theme_long = build_theme_long(projects)

if SELECTED_THEME_CODES:
    selected_theme_codes = list(dict.fromkeys(SELECTED_THEME_CODES))
else:
    auto_candidates = theme_ranking.loc[
        theme_ranking['ranking_eligible'].astype(bool)
    ].sort_values('overall_rank')
    selected_theme_codes = auto_candidates['theme_code'].head(AUTO_TOP_THEME_N).tolist()
    print('SELECTED_THEME_CODESが空のため、2_02の上位テーマを使用:', selected_theme_codes)

unknown = sorted(set(selected_theme_codes) - set(theme_long['theme_code']))
if unknown:
    raise ValueError(f'観測されていないthemeが指定されています: {unknown}')
selected_theme_summary = theme_ranking.loc[
    theme_ranking['theme_code'].isin(selected_theme_codes)
].sort_values('overall_rank')
display(merge_diagnostics.to_frame())
display(selected_theme_summary[[
    'overall_rank', 'theme_code', 'theme_label', 'high_app_project_count',
    'high_app_rate', 'ministry_breadth', 'domain_breadth', 'aiu_fit', 'overall_score'
]])

## 3. テーマ内のpolicy domainを順位付け

順位はテーマごとに計算します。初期重みはHigh-app件数25%、平滑化率15%、省庁breadth15%、低省庁集中10%、High-app予算15%、直近High-app件数10%、課題文充足率10%です。AIU Fitは同一テーマ内で一定なので、ドメイン順位には重ねて入れません。

In [ ]:
domain_metrics = compute_theme_domain_metrics(
    theme_long, selected_theme_codes,
    budget_col=BUDGET_COL, issue_col=ISSUE_COL,
    recent_years=RECENT_YEARS, prior_strength=RATE_PRIOR_STRENGTH,
)
domain_scored, effective_weights = score_theme_domain_metrics(
    domain_metrics, weights=DOMAIN_WEIGHTS
)
domain_yearly = compute_theme_domain_yearly_metrics(theme_long, selected_theme_codes)
ministry_opportunities = compute_ministry_domain_opportunities(
    theme_long, selected_theme_codes, budget_col=BUDGET_COL
)
top_domain_rows = domain_scored.loc[
    domain_scored['domain_rank_within_theme'].notna()
    & domain_scored['domain_rank_within_theme'].le(TOP_DOMAINS_PER_THEME)
].copy()
if top_domain_rows.empty:
    raise RuntimeError('選定テーマにHigh-app事業がなく、ドメイン候補を作れません。')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DOMAIN_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
domain_scored.to_csv(DOMAIN_METRICS_PATH, index=False)
domain_yearly.to_csv(DOMAIN_YEARLY_PATH, index=False)
ministry_opportunities.to_csv(MINISTRY_OPPORTUNITY_PATH, index=False)
effective_weights.to_frame().to_csv(OUTPUT_DIR / 'domain_effective_weights.csv')
print('saved:', DOMAIN_METRICS_PATH)
print('saved:', DOMAIN_YEARLY_PATH)
print('saved:', MINISTRY_OPPORTUNITY_PATH)
display(pd.concat([
    pd.Series(DOMAIN_WEIGHTS, name='configured_weight'), effective_weights
], axis=1))

In [ ]:
domain_table_columns = [
    'theme_code', 'domain_rank_within_theme', 'policy_domain_label',
    'project_count', 'high_app_project_count', 'high_app_rate',
    'high_app_rate_smoothed', 'ministry_breadth', 'ministry_hhi',
    'high_app_budget_total', 'high_app_budget_median',
    'high_app_budget_hhi', 'high_app_top3_budget_share',
    'recent_high_app_project_count', 'high_app_rate_momentum',
    'issue_coverage_rate', 'domain_opportunity_score',
]
display(top_domain_rows[domain_table_columns].style.format({
    'high_app_rate': '{:.1%}', 'high_app_rate_smoothed': '{:.1%}',
    'ministry_hhi': '{:.3f}', 'high_app_budget_total': '{:,.0f}',
    'high_app_budget_median': '{:,.0f}', 'high_app_budget_hhi': '{:.3f}',
    'high_app_top3_budget_share': '{:.1%}', 'high_app_rate_momentum': '{:+.1%}',
    'issue_coverage_rate': '{:.1%}', 'domain_opportunity_score': '{:.1f}',
}).background_gradient(subset=['domain_opportunity_score'], cmap='YlGn'))

## 4. テーマ×ドメインの機会マップ

市場量、適用率、横展開性、予算、直近性を別々に見ます。総合スコアだけで判断せず、各ヒートマップの形を比較してください。

In [ ]:
opportunity_heatmaps = [
    ('high_app_project_count', 'High-app件数', 'Blues', None, None),
    ('high_app_rate', 'High-app率', 'YlGn', 0, 1),
    ('ministry_breadth', '省庁breadth', 'PuBu', None, None),
    ('ministry_hhi', '省庁集中HHI', 'Reds', 0, 1),
    ('recent_high_app_project_count', f'直近{RECENT_YEARS}年High-app件数', 'BuPu', None, None),
    ('domain_opportunity_score', 'ドメイン機会スコア', 'YlOrRd', 0, 100),
]
fig, axes = plt.subplots(3, 2, figsize=(22, 18))
for ax, (column, title, cmap, vmin, vmax) in zip(axes.flat, opportunity_heatmaps):
    pivot = domain_scored.pivot(index='theme_code', columns='policy_domain', values=column).reindex(
        index=selected_theme_codes, columns=list(POLICY_DOMAINS)
    )
    sns.heatmap(
        pivot, mask=pivot.isna(), cmap=cmap, vmin=vmin, vmax=vmax,
        annot=True, fmt='.1f', linewidths=0.4, ax=ax, cbar_kws={'shrink': 0.75}
    )
    ax.set_title(title)
    ax.set_xlabel('policy_domain')
    ax.set_ylabel('theme')
plt.tight_layout()
save_figure('theme_domain_opportunity_heatmaps.png')
plt.show()

In [ ]:
n_themes = len(selected_theme_codes)
fig, axes = plt.subplots(n_themes, 1, figsize=(14, max(7, 6 * n_themes)), squeeze=False)
last_scatter = None
for ax, theme_code in zip(axes.flat, selected_theme_codes):
    part = domain_scored.loc[
        domain_scored['theme_code'].eq(theme_code) & domain_scored['ranking_eligible']
    ].copy()
    budget_size = np.log1p(part['high_app_budget_total'].clip(lower=0))
    if budget_size.notna().any() and budget_size.max() > 0:
        sizes = 120 + 700 * budget_size.fillna(0) / budget_size.max()
        size_note = 'バブルサイズ=log1p(High-app予算)'
    else:
        sizes = 120 + 100 * part['ministry_breadth']
        size_note = 'バブルサイズ=省庁breadth（予算なし）'
    last_scatter = ax.scatter(
        part['high_app_rate_smoothed'], part['high_app_project_count'],
        s=sizes, c=part['domain_opportunity_score'], cmap='viridis',
        vmin=0, vmax=100, alpha=0.75, edgecolor='black', linewidth=0.5,
    )
    for row in part.itertuples():
        ax.annotate(row.policy_domain, (row.high_app_rate_smoothed, row.high_app_project_count),
                    xytext=(5, 5), textcoords='offset points')
    ax.xaxis.set_major_formatter(PercentFormatter(1.0))
    ax.set_title(f'{theme_code}: ドメイン候補（{size_note}）')
    ax.set_xlabel('High-app rate（平滑化）')
    ax.set_ylabel('High-app project count')
if last_scatter is not None:
    fig.colorbar(last_scatter, ax=axes.ravel().tolist(), label='domain opportunity score', shrink=0.7)
plt.tight_layout()
save_figure('domain_count_rate_budget_bubbles.png')
plt.show()

## 5. 上位ドメインの評価プロファイル

同じ順位でも、案件量型・高適用率型・省庁横展開型・予算型など性格が異なります。percentile scoreの内訳から提案戦略を分けます。

In [ ]:
domain_component_labels = {
    'high_app_project_count_score': 'High-app件数',
    'high_app_rate_smoothed_score': 'High-app率',
    'ministry_breadth_score': '省庁breadth',
    'low_ministry_concentration_score': '低集中度',
    'high_app_budget_total_score': '予算',
    'recent_high_app_project_count_score': '直近性',
    'issue_coverage_rate_score': '課題文充足',
}
profile = top_domain_rows.assign(
    segment=lambda frame: frame['theme_code'] + ' × ' + frame['policy_domain']
).set_index('segment')[list(domain_component_labels)].rename(columns=domain_component_labels)
plt.figure(figsize=(13, max(6, len(profile) * 0.55)))
sns.heatmap(profile, cmap='YlGnBu', vmin=0, vmax=100, annot=True, fmt='.0f')
plt.title('上位テーマ×ドメインの機会プロファイル')
plt.xlabel('')
plt.ylabel('theme × domain')
plt.tight_layout()
save_figure('top_domain_opportunity_profiles.png')
plt.show()

## 6. 年度推移とモメンタム

上位ドメインが直近にも継続しているかを確認します。件数0の年度は0、該当事業がない年度の率は欠損です。

In [ ]:
top_pairs = top_domain_rows[['theme_code', 'policy_domain']].drop_duplicates()
trend = domain_yearly.merge(top_pairs, on=['theme_code', 'policy_domain'], how='inner')
trend['segment'] = trend['theme_code'] + ' × ' + trend['policy_domain']
fig, axes = plt.subplots(2, 1, figsize=(15, 13), sharex=True)
sns.lineplot(
    data=trend, x=YEAR_COL, y='high_app_project_count', hue='segment', marker='o', ax=axes[0]
)
sns.lineplot(
    data=trend, x=YEAR_COL, y='high_app_rate', hue='segment', marker='o', ax=axes[1], legend=False
)
axes[0].set_title('上位テーマ×ドメインの年度別High-app件数')
axes[1].set_title('上位テーマ×ドメインの年度別High-app率')
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
save_figure('top_domain_yearly_trends.png')
plt.show()

## 7. 省庁アカウントマップ

どの省庁が各ドメインのHigh-app案件を多く持つかを確認し、ヒアリング・営業・共同検討の候補を作ります。省庁欠損は除外します。

In [ ]:
account_top = ministry_opportunities.loc[
    ministry_opportunities['high_app_project_count'].gt(0)
].sort_values(
    ['theme_code', 'policy_domain', 'high_app_project_count', 'high_app_budget_total'],
    ascending=[True, True, False, False],
)
account_top.to_csv(OUTPUT_DIR / 'ministry_account_shortlist.csv', index=False)
display(account_top.head(50))

for theme_code in selected_theme_codes:
    part = account_top.loc[account_top['theme_code'].eq(theme_code)]
    if part.empty:
        continue
    top_ministries = part.groupby('responsible_ministry')['high_app_project_count'].sum().nlargest(15).index
    pivot = part.loc[part['responsible_ministry'].isin(top_ministries)].pivot_table(
        index='responsible_ministry', columns='policy_domain',
        values='high_app_project_count', aggfunc='sum', fill_value=0,
    )
    plt.figure(figsize=(12, max(5, len(pivot) * 0.55)))
    sns.heatmap(pivot, cmap='Blues', annot=True, fmt='.0f', linewidths=0.5)
    plt.title(f'{theme_code}: 省庁 × policy domain High-app件数')
    plt.xlabel('policy_domain')
    plt.ylabel('responsible_ministry')
    plt.tight_layout()
    save_figure(f'ministry_domain_{theme_code}.png')
    plt.show()

## 8. 予算規模と大型案件依存

予算は市場規模そのものではありませんが、事業規模の参考になります。合計・中央値に加え、予算HHIと上位3事業シェアを確認し、少数大型案件だけで高く見えていないかを診断します。負値は欠損相当として除外します。

In [ ]:
budget_diagnostics = top_domain_rows[[
    'theme_code', 'policy_domain_label', 'high_app_project_count',
    'high_app_budget_total', 'high_app_budget_median', 'high_app_budget_nonmissing_rate',
    'high_app_budget_hhi', 'high_app_top3_budget_share',
]].copy()
display(budget_diagnostics.style.format({
    'high_app_budget_total': '{:,.0f}', 'high_app_budget_median': '{:,.0f}',
    'high_app_budget_nonmissing_rate': '{:.1%}', 'high_app_budget_hhi': '{:.3f}',
    'high_app_top3_budget_share': '{:.1%}',
}))

high_top_projects = theme_long.loc[theme_long['is_high_app']].merge(
    top_pairs, on=['theme_code', 'policy_domain'], how='inner', validate='many_to_one'
).copy()
if BUDGET_COL in high_top_projects:
    high_top_projects['budget_numeric'] = pd.to_numeric(high_top_projects[BUDGET_COL], errors='coerce').where(lambda s: s.ge(0))
    budget_plot = high_top_projects.dropna(subset=['budget_numeric']).copy()
    if not budget_plot.empty:
        budget_plot['log1p_budget'] = np.log1p(budget_plot['budget_numeric'])
        budget_plot['segment'] = budget_plot['theme_code'] + ' × ' + budget_plot['policy_domain']
        plt.figure(figsize=(15, max(7, budget_plot['segment'].nunique() * 0.6)))
        sns.boxplot(data=budget_plot, x='log1p_budget', y='segment', showfliers=True)
        plt.title('上位テーマ×ドメインのHigh-app予算分布（log1p）')
        plt.tight_layout()
        save_figure('top_domain_budget_distribution.png')
        plt.show()
    else:
        print('有効な予算値がないため予算分布図を省略します。')

## 9. 課題文から特徴的なフレーズを抽出

上位テーマ×ドメインの4テキスト列から、他セグメントより相対的に強い日本語char n-gramを抽出します。これはニーズ仮説を作るための探索的証拠であり、因果的重要度ではありません。一般語や不自然な断片は、元事業と照合して人間が除外してください。

In [ ]:
phrases = pd.DataFrame()
phrase_pairs = top_pairs.head(MAX_PHRASE_PAIRS)
if RUN_TEXT_PHRASE_ANALYSIS:
    try:
        phrases = extract_distinctive_char_ngrams(
            theme_long, phrase_pairs, top_n=TOP_PHRASES, min_df=2,
            text_cols=('project_name', 'project_objective', 'project_summary', 'current_issues'),
        )
    except (ImportError, ValueError) as exc:
        print('フレーズ抽出を省略:', exc)
if not phrases.empty:
    phrases.to_csv(PHRASE_PATH, index=False)
    display(phrases.head(50))
    for (theme_code, domain), part in phrases.groupby(['theme_code', 'policy_domain'], observed=True):
        plot_part = part.nlargest(TOP_PHRASES, 'distinctiveness').sort_values('distinctiveness')
        plt.figure(figsize=(10, max(5, len(plot_part) * 0.35)))
        plt.barh(plot_part['phrase'], plot_part['distinctiveness'])
        plt.title(f'{theme_code} × {domain}: 特徴フレーズ')
        plt.xlabel('mean TF-IDF差（当該segment − その他）')
        plt.tight_layout()
        save_figure(f'distinctive_phrases_{theme_code}_{domain}.png')
        plt.show()
else:
    print('保存する特徴フレーズはありません。')

## 10. 代表事業と現状課題を確認

分類理由、事業名、現状課題を読み、同じテーマ・ドメイン内に共通する業務課題が本当に存在するかを確認します。予算がある場合は大型案件と最新案件を優先表示します。

In [ ]:
representative = high_top_projects.copy()
if 'budget_numeric' not in representative:
    representative['budget_numeric'] = np.nan
representative = representative.sort_values(
    ['theme_code', 'policy_domain', 'budget_numeric', YEAR_COL],
    ascending=[True, True, False, False], na_position='last',
).drop_duplicates('project_key')
review_columns = [column for column in [
    'theme_code', 'policy_domain', 'source_split', ID_COL, YEAR_COL,
    'project_name', 'responsible_ministry', BUDGET_COL,
    'current_issues', 'project_objective', 'classification_reason',
] if column in representative.columns]
representative[review_columns].to_csv(OUTPUT_DIR / 'representative_high_app_projects.csv', index=False)
for (theme_code, domain), part in representative.groupby(['theme_code', 'policy_domain'], observed=True):
    print(f'--- {theme_code} × {domain}: representative projects ---')
    display(part[review_columns].head(10))

## 11. サービス・コンサルティング案

下表の`observed_evidence`だけがデータ集計です。それ以外は、admin_process別の業務改革観点とai_usecase別のサービスカタログを組み合わせた初期仮説です。ヒアリング前の提案骨子として使い、対象省庁の制度・データ・責任分界に合わせて更新します。

In [ ]:
service_cards = build_service_opportunity_cards(
    domain_scored, top_domains_per_theme=TOP_DOMAINS_PER_THEME,
    admin_catalog=ADMIN_SERVICE_RULES, usecase_catalog=USECASE_SERVICE_RULES,
)
service_cards.to_csv(OUTPUT_DIR / 'service_opportunity_cards.csv', index=False)
display(service_cards[[
    'theme_code', 'policy_domain_label', 'domain_rank_within_theme',
    'recommended_offer', 'observed_evidence', 'consulting_entry',
    'discovery_and_assessment', 'poc_deliverable', 'implementation_support',
    'governance_and_risks', 'kpi_hypotheses',
]])

In [ ]:
offer_matrix = service_cards.pivot_table(
    index=['theme_code', 'policy_domain'],
    values=['consulting_entry', 'poc_deliverable', 'implementation_support', 'governance_and_risks', 'kpi_hypotheses'],
    aggfunc='first',
)
display(offer_matrix)

if account_top.empty:
    priority_accounts = pd.DataFrame(columns=[
        'theme_code', 'policy_domain', 'priority_ministries', 'top_ministry_high_app_count'
    ])
else:
    priority_accounts = account_top.groupby(
        ['theme_code', 'policy_domain'], as_index=False
    ).head(3)[[
        'theme_code', 'policy_domain', 'responsible_ministry', 'high_app_project_count'
    ]].groupby(['theme_code', 'policy_domain']).agg({
        'responsible_ministry': lambda values: ' / '.join(map(str, values)),
        'high_app_project_count': 'sum',
    }).rename(columns={
        'responsible_ministry': 'priority_ministries',
        'high_app_project_count': 'top_ministry_high_app_count',
    }).reset_index()
portfolio = service_cards.merge(
    priority_accounts,
    on=['theme_code', 'policy_domain'], how='left', validate='one_to_one',
)
portfolio.to_csv(OUTPUT_DIR / 'service_portfolio_shortlist.csv', index=False)
display(portfolio[[
    'theme_code', 'policy_domain_label', 'domain_rank_within_theme',
    'recommended_offer', 'priority_ministries', 'observed_evidence',
    'poc_deliverable', 'governance_and_risks', 'kpi_hypotheses',
]])

## 12. 最終検討チェックリスト

上位候補ごとに次を確認して、提供領域とサービスを確定します。

1. **Problem**: 代表事業の`current_issues`に共通課題が実在するか
2. **Customer**: 重点省庁・原課・利用者・最終意思決定者は誰か
3. **Data**: 必要データ、ラベル、更新頻度、個人情報、システム連携は何か
4. **Decision**: AIは助言か自動処理か。Human-in-the-loopと異議申立てをどう置くか
5. **Value**: 時間、品質、件数、政策効果のどれをKPIにするか
6. **Delivery**: 診断、PoC、本番実装、運用、人材育成のどこから入るか
7. **Repeatability**: 他省庁・他domainへ共通化できるデータモデルや業務部品は何か
8. **Risk**: 公平性、説明責任、セキュリティ、調達、運用責任を誰が持つか

Notebookの順位は候補抽出です。最終提案は、代表事業レビューと省庁ヒアリングで検証してください。